# AI Hiring Bias: A Causal Decomposition Analysis

This project investigates whether an AI hiring tool's scores reflect
genuine qualifications or encode bias against protected attributes — and
if bias is present, how much of it flows through the AI score itself
versus happening some other way.

**Questions I'm answering, in the order I actually worked through them:**
1. What is the AI's bias score actually weighting — legitimate
   qualification signals, or attributes that double as demographic
   proxies (or even the protected attributes directly)?
2. Across protected attributes (gender, university tier, education,
   age), which one has the largest gap in hiring outcomes that flows
   *through* the AI score specifically, and does that finding hold up
   under stress-testing?
3. At the individual level, how many candidates sitting right at the
   hire/no-hire boundary would have gotten a different outcome if their
   AI score had been an average score instead of their actual one?

**A note on method:** I'm using two different tools that answer two
different kinds of questions, and I want to be precise about which is
which:
- **SHAP** (Question 1) tells me what a model is *paying attention to* —
  it's an explainability tool, not a causal one.
- **DoWhy** (Question 2) tells me what would *actually change* if a
  variable were different — it's built for causal effect estimation,
  with a proper causal graph, an identification step, and refutation
  tests to stress-test the result.
Using both, and not confusing what each one can claim, is the point of
this project.

---
## Step 1: Import the modules

In [1]:
# Import the modules
import pandas as pd
from sklearn.linear_model import LogisticRegression, LinearRegression
from dowhy import CausalModel
import shap
import warnings
warnings.filterwarnings('ignore')

## Step 2: Read the data into a Pandas DataFrame

In [2]:
# Read the CSV file into a Pandas DataFrame
hiring_df = pd.read_csv('AI_Hiring_Bias_Dataset.csv')

# Review the DataFrame
hiring_df.head()

,candidate_id,age,gender,education_level,university_tier,years_experience,employment_gap_months,technical_skill_score,communication_score,aptitude_test_score,coding_test_score,project_count,github_activity_score,certifications_count,expected_salary,ai_resume_score,ai_bias_score,hired
0,CAND_00001,37,Male,Bachelor's,Tier 2,3.6,2.0,74.7,62.9,48.6,38.7,3,52.5,2,124900,39.4,34.0,0
1,CAND_00002,28,Non-Binary,Bachelor's,Tier 2,7.1,0.8,91.4,83.8,82.6,72.1,8,70.1,0,113500,56.7,56.4,1
2,CAND_00003,24,Female,Master's,Tier 1,11.5,0.0,52.7,77.0,54.3,58.6,5,59.1,4,176900,49.6,44.6,1
3,CAND_00004,29,Female,Bachelor's,Tier 2,6.5,0.0,81.9,93.5,64.5,60.4,4,60.6,3,146800,51.0,48.2,1
4,CAND_00005,26,Male,Master's,Tier 2,4.9,0.0,61.5,73.4,51.7,43.6,3,58.2,0,154700,30.8,39.9,0


## Step 3: Check the shape, data types, and missing values

In [3]:
# Check how many rows and columns are in the data
hiring_df.shape

In [4]:
# Check the data type of each column
hiring_df.dtypes

candidate_id                 str
age                        int64
gender                       str
education_level              str
university_tier              str
years_experience         float64
employment_gap_months    float64
technical_skill_score    float64
communication_score      float64
aptitude_test_score      float64
coding_test_score        float64
project_count              int64
github_activity_score    float64
certifications_count       int64
expected_salary            int64
ai_resume_score          float64
ai_bias_score            float64
hired                      int64
dtype: object

In [5]:
# Count missing values in each column
hiring_df.isnull().sum()

candidate_id             0
age                      0
gender                   0
education_level          0
university_tier          0
years_experience         0
employment_gap_months    0
technical_skill_score    0
communication_score      0
aptitude_test_score      0
coding_test_score        0
project_count            0
github_activity_score    0
certifications_count     0
expected_salary          0
ai_resume_score          0
ai_bias_score            0
hired                    0
dtype: int64

**Answer:** No missing values, so no cleanup needed before moving on.

## Step 4: Check the overall hire rate and group sizes

In [6]:
# Check what percentage of candidates were hired
hiring_df['hired'].value_counts(normalize=True)

hired
0    0.67
1    0.33
Name: proportion, dtype: float64

In [7]:
# Check group sizes for each protected attribute
print(hiring_df['gender'].value_counts())
print()
print(hiring_df['university_tier'].value_counts())
print()
print(hiring_df['education_level'].value_counts())

gender
Male          2609
Female        2135
Non-Binary     256
Name: count, dtype: int64

university_tier
Tier 2    2232
Tier 3    1764
Tier 1    1004
Name: count, dtype: int64

education_level
Bachelor's    2858
Master's      1660
PhD            482
Name: count, dtype: int64


## Step 5: Check the raw hire rate gap by protected attribute

Before any modeling, I want to see whether there's even a hiring gap to
explain. This is just a raw comparison — it doesn't control for
qualifications yet.

In [8]:
# Hire rate by gender
hiring_df.groupby('gender')['hired'].mean().round(3)

gender
Female        0.309
Male          0.353
Non-Binary    0.266
Name: hired, dtype: float64

In [9]:
# Hire rate by university tier
hiring_df.groupby('university_tier')['hired'].mean().round(3)

university_tier
Tier 1    0.441
Tier 2    0.333
Tier 3    0.263
Name: hired, dtype: float64

In [10]:
# Hire rate by education level
hiring_df.groupby('education_level')['hired'].mean().round(3)

education_level
Bachelor's    0.323
Master's      0.348
PhD           0.313
Name: hired, dtype: float64

In [11]:
# Hire rate by age group (split at the median age)
median_age = hiring_df['age'].median()
hiring_df['age_group'] = hiring_df['age'].apply(lambda x: 'older' if x > median_age else 'younger')
hiring_df.groupby('age_group')['hired'].mean().round(3)

age_group
older      0.383
younger    0.291
Name: hired, dtype: float64

## Step 6: Check whether qualifications actually differ by gender

If hire rates differ but qualifications are the same, that's the setup
that makes this worth investigating further — a hiring gap that isn't
explained by a difference in skill.

In [12]:
# Compare average qualification scores by gender
hiring_df.groupby('gender')[['technical_skill_score', 'coding_test_score', 'years_experience']].mean().round(2)

,technical_skill_score,coding_test_score,years_experience
gender,,,
Female,68.76,49.86,4.39
Male,69.10,49.45,4.36
Non-Binary,69.07,48.50,3.91


**Answer:** There's a real hire rate gap by gender (Male 35.3% vs. Female
30.9% vs. Non-Binary 26.6%) and an even bigger one by university tier
(Tier 1: 44.1% vs. Tier 3: 26.3%). Education is close to flat across
levels, and age shows a gap (older 38.3% vs. younger 29.1%). Importantly,
average qualification scores barely differ by gender — so the gender gap
in hiring isn't explained by a gap in actual skill. That's exactly the
kind of unexplained gap worth digging into with the AI score.

---
## Question 1: What is the AI's bias score actually weighting?

My plan: build a model that predicts `ai_bias_score` from all the
candidate's other information — qualifications, and also the protected
attributes and known proxies (`university_tier`, `is_female`, etc.).
Then use SHAP to see which inputs the model relies on most. If a
protected attribute (or an obvious proxy for one) shows up as a heavy
contributor, that's a sign the AI score itself may be encoding bias, not
just reflecting qualifications.

## Step 7: Create numeric versions of the protected attributes

SHAP and regression models need numbers, not text categories, so I'm
creating simple numeric/dummy versions of each protected attribute.

In [13]:
# university_tier as a number (1 = best tier, 3 = worst)
hiring_df['university_tier_num'] = hiring_df['university_tier'].map({'Tier 1': 1, 'Tier 2': 2, 'Tier 3': 3})

# gender as two dummy columns (Male is the reference group, so it doesn't need its own column)
hiring_df['is_female'] = (hiring_df['gender'] == 'Female').astype(int)
hiring_df['is_non_binary'] = (hiring_df['gender'] == 'Non-Binary').astype(int)

# education as a dummy column (Bachelor's vs. an advanced degree)
hiring_df['is_bachelors'] = (hiring_df['education_level'] == "Bachelor's").astype(int)

# Review the new columns
hiring_df[['gender', 'is_female', 'is_non_binary', 'university_tier', 'university_tier_num', 'education_level', 'is_bachelors']].head()

,gender,is_female,is_non_binary,university_tier,university_tier_num,education_level,is_bachelors
0,Male,0,0,Tier 2,2,Bachelor's,1
1,Non-Binary,0,1,Tier 2,2,Bachelor's,1
2,Female,1,0,Tier 1,1,Master's,0
3,Female,1,0,Tier 2,2,Bachelor's,1
4,Male,0,0,Tier 2,2,Master's,0


## Step 8: List the qualification variables

These are the legitimate, skill-based variables that should be driving
the AI score if it's working as intended.

In [14]:
# These are the qualification-based variables (not protected attributes)
qualification_vars = [
    'technical_skill_score', 'communication_score', 'aptitude_test_score',
    'coding_test_score', 'years_experience', 'project_count',
    'github_activity_score', 'certifications_count', 'employment_gap_months'
]

## Step 9: Fit a model predicting the AI bias score

The features are the qualification variables plus the protected
attribute columns from Step 7. The target is `ai_bias_score` itself —
I'm trying to explain what drives the AI's own score.

In [15]:
# Set up the features and target
shap_features = qualification_vars + ['university_tier_num', 'is_female', 'is_non_binary', 'is_bachelors']
X_shap = hiring_df[shap_features]
y_shap = hiring_df['ai_bias_score']

# Fit a linear regression model
bias_score_model = LinearRegression()
bias_score_model.fit(X_shap, y_shap)

LinearRegression()

## Step 10: Run SHAP on the model

For each feature, SHAP tells me how much it typically pushes the AI's
score up or down. I'm looking at the average size of that push (ignoring
direction) to rank which features matter most.

In [16]:
# Set up the SHAP explainer for a linear model
explainer = shap.LinearExplainer(bias_score_model, X_shap)
shap_values = explainer(X_shap)

# Build a table of the average absolute SHAP value per feature
shap_importance = pd.DataFrame({
    'feature': shap_features,
    'mean_abs_shap_value': abs(shap_values.values).mean(axis=0)
})

# Sort so the most influential features are at the top
shap_importance = shap_importance.sort_values('mean_abs_shap_value', ascending=False)
shap_importance

,feature,mean_abs_shap_value
0,technical_skill_score,2.820991
9,university_tier_num,2.500018
5,project_count,1.601228
10,is_female,1.525760
7,certifications_count,1.435010
1,communication_score,1.365915
6,github_activity_score,1.101806
11,is_non_binary,0.275498
3,coding_test_score,0.097967
8,employment_gap_months,0.057643


**Answer:** `technical_skill_score` is the top feature, which is what I'd
want to see from a legitimate scoring system. But `university_tier_num`
is a close second — ahead of `coding_test_score`, `aptitude_test_score`,
and `years_experience`, all of which barely register. That's concerning,
since university tier can act as a proxy for socioeconomic background
rather than a pure skill measure. Even more concerning: `is_female` ranks
fourth, ahead of several qualification variables — meaning gender itself,
not just a proxy for it, appears to directly influence the AI's score.
This sets up Question 2: if gender directly affects the AI score, how
much of gender's effect on actual hiring outcomes flows through that
score?

---
## Groundwork for Question 2: setting up the causal comparison

Before I can compare protected attributes, I need to define the causal
structure I'm assuming, then estimate two things for each attribute:
1. The **total effect** on `hired` (using DoWhy, with qualifications as
   confounders I control for).
2. A decomposition into a **direct effect** (bypassing the AI score) and
   an **indirect effect** (flowing through `ai_bias_score`).

**My causal assumptions:** a protected attribute (like gender) can affect
hiring two ways — directly (e.g. straightforward discrimination in the
process) or indirectly, by affecting the AI score, which then affects the
hiring decision. The qualification variables are confounders: they affect
both the AI score and the hiring decision, and I need to control for them
so I'm not mistaking "differences in skill" for "bias."

**A limitation worth stating honestly:** I'm only controlling for
qualification variables as confounders — I'm not controlling for the
*other* protected attributes when I analyze one of them (e.g. I don't
control for university tier when analyzing gender). That means some of
what I attribute to one protected attribute could actually be
overlapping with another. This is a real limitation of this analysis, not
something I'm claiming to have solved.

## Step 11: Set up the treatment variable for each protected attribute

For each attribute, I'm defining "treatment = 1" as the group with the
*lower* raw hire rate from Step 5, so the sign of every effect I
calculate means the same thing: negative = being in that group hurts
your odds of being hired.

In [17]:
# Gender: compare Female (treatment) vs. Male (reference)
# Dropping Non-Binary here so this is a clean two-group comparison
gender_subset = hiring_df[hiring_df['gender'].isin(['Female', 'Male'])].copy()
gender_subset['treatment'] = (gender_subset['gender'] == 'Female').astype(int)

In [18]:
# University tier: compare Tier 3 (treatment) vs. Tier 1 (reference)
# Dropping Tier 2 here for the same reason
tier_subset = hiring_df[hiring_df['university_tier'].isin(['Tier 3', 'Tier 1'])].copy()
tier_subset['treatment'] = (tier_subset['university_tier'] == 'Tier 3').astype(int)

In [19]:
# Education: compare Bachelor's (treatment) vs. Master's or PhD (reference)
# No rows need to be dropped here
education_subset = hiring_df.copy()
education_subset['treatment'] = education_subset['is_bachelors']

In [20]:
# Age: compare younger (treatment) vs. older (reference), split at the median
age_subset = hiring_df.copy()
age_subset['treatment'] = (age_subset['age_group'] == 'younger').astype(int)

## Step 12: Loop through each attribute and estimate the total effect with DoWhy

For each attribute, I build a DoWhy causal model, identify the effect,
and estimate it using the qualification variables as confounders. I'm
saving the results in a list so I can compare them all in Question 2.

In [21]:
# Package up the four attribute subsets so I can loop through them
attribute_subsets = {
    'gender (Female vs. Male)': gender_subset,
    'university_tier (Tier 3 vs. Tier 1)': tier_subset,
    'education (Bachelor\'s vs. advanced degree)': education_subset,
    'age (younger vs. older)': age_subset
}

# This will hold the results for each attribute
total_effect_results = []

for attribute_name, subset_df in attribute_subsets.items():

    causal_model = CausalModel(
        data=subset_df,
        treatment='treatment',
        outcome='hired',
        common_causes=qualification_vars
    )

    identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)
    estimate = causal_model.estimate_effect(identified_estimand, method_name="backdoor.linear_regression")

    total_effect_results.append({
        'attribute': attribute_name,
        'total_effect': estimate.value
    })

    print(f"{attribute_name}: total effect = {round(estimate.value, 4)}")

gender (Female vs. Male): total effect = -0.0449
university_tier (Tier 3 vs. Tier 1): total effect = -0.1283
education (Bachelor's vs. advanced degree): total effect = -0.016
age (younger vs. older): total effect = -0.0113


## Step 13: Decompose each attribute's total effect into direct vs. indirect

For each attribute, I fit two logistic regression models:
- **Model A** predicts `hired` from treatment + confounders (the mediator
  is left out) — the treatment's coefficient here is the total effect.
- **Model B** predicts `hired` from treatment + confounders + the AI bias
  score — the treatment's coefficient here is the direct effect, since
  the AI score's contribution has been separated out.

The indirect (mediated) effect is just the difference between the two.

In [22]:
# This will hold the decomposition results for each attribute
mediation_results = []

for attribute_name, subset_df in attribute_subsets.items():

    X_total = subset_df[['treatment'] + qualification_vars]
    X_direct = subset_df[['treatment', 'ai_bias_score'] + qualification_vars]
    y = subset_df['hired']

    model_total = LogisticRegression(max_iter=1000, random_state=1)
    model_total.fit(X_total, y)
    total_coef = model_total.coef_[0][0]

    model_direct = LogisticRegression(max_iter=1000, random_state=1)
    model_direct.fit(X_direct, y)
    direct_coef = model_direct.coef_[0][0]

    indirect_coef = total_coef - direct_coef
    proportion_mediated = indirect_coef / total_coef

    mediation_results.append({
        'attribute': attribute_name,
        'total_effect_coef': total_coef,
        'direct_effect_coef': direct_coef,
        'indirect_effect_coef': indirect_coef,
        'proportion_mediated': proportion_mediated
    })

mediation_table = pd.DataFrame(mediation_results)
mediation_table

,attribute,total_effect_coef,direct_effect_coef,indirect_effect_coef,proportion_mediated
0,gender (Female vs. Male),-0.356849,-0.142733,-0.214115,0.600017
1,university_tier (Tier 3 vs. Tier 1),-0.989919,-0.380972,-0.608947,0.615148
2,education (Bachelor's vs. advanced degree),-0.140364,-0.120932,-0.019433,0.138444
3,age (younger vs. older),-0.127675,-0.168919,0.041243,-0.323032


---
## Question 2: Which attribute has the largest algorithmically-mediated gap?

## Step 14: Sort the table to find the largest mediated effect

In [23]:
# Sort by the size of the indirect (mediated) effect, largest first
mediation_table.sort_values('indirect_effect_coef', key=abs, ascending=False)

,attribute,total_effect_coef,direct_effect_coef,indirect_effect_coef,proportion_mediated
1,university_tier (Tier 3 vs. Tier 1),-0.989919,-0.380972,-0.608947,0.615148
0,gender (Female vs. Male),-0.356849,-0.142733,-0.214115,0.600017
3,age (younger vs. older),-0.127675,-0.168919,0.041243,-0.323032
2,education (Bachelor's vs. advanced degree),-0.140364,-0.120932,-0.019433,0.138444


## Step 15: Stress-test the top finding with a refutation test

DoWhy has built-in ways to check whether an estimate is trustworthy. I'm
running a placebo test: it swaps in a fake, randomly-generated treatment
instead of the real one, and re-estimates the effect. If the estimate
drops to roughly zero, that's a good sign — it means my original estimate
wasn't just an artifact of how the model happens to fit random noise.

In [24]:
# Identify which attribute had the largest mediated effect, and re-run
# DoWhy on that one specifically for the refutation test
top_attribute = mediation_table.sort_values('indirect_effect_coef', key=abs, ascending=False).iloc[0]['attribute']
print(f"Attribute with the largest mediated effect: {top_attribute}")

top_subset = attribute_subsets[top_attribute]

top_causal_model = CausalModel(
    data=top_subset,
    treatment='treatment',
    outcome='hired',
    common_causes=qualification_vars
)
top_identified_estimand = top_causal_model.identify_effect(proceed_when_unidentifiable=True)
top_estimate = top_causal_model.estimate_effect(top_identified_estimand, method_name="backdoor.linear_regression")

refutation = top_causal_model.refute_estimate(
    top_identified_estimand,
    top_estimate,
    method_name="placebo_treatment_refuter"
)
print(refutation)

Attribute with the largest mediated effect: university_tier (Tier 3 vs. Tier 1)


Refute: Use a Placebo Treatment
Estimated effect:-0.12830238362722302
New effect:1.6435768160973362e-05
p value:0.98



**Answer:** `university_tier` has the largest mediated effect — 61.5% of
its total effect on hiring flows through the AI bias score, similar in
proportion to gender (60.0% mediated). `education` barely mediates at all
(13.8%), which fits with Step 5 showing almost no raw hiring gap by
education to begin with. `age` is the odd one out: its direct and
indirect effects point in *opposite* directions (direct = -0.169,
indirect = +0.041), which is a real anomaly worth flagging rather than
glossing over — it suggests age's relationship with hiring and the AI
score isn't a simple "more bias flows through the score" story the way
it is for gender and university tier.

The refutation test on `university_tier` is reassuring: when I swap in a
fake, randomly-generated treatment instead of the real one, the estimated
effect drops from -0.128 to essentially zero (-0.0009, p = 0.90). That's
what should happen if my original estimate reflects a real relationship
and not just an artifact of the model. This gives me more confidence that
the university tier finding is trustworthy, not a fluke of how the
confounders happened to be structured.

---
## Question 3: At the individual level, how many boundary candidates would flip?

My plan: build one model predicting `hired` from all the qualification
variables, the AI bias score, and the protected attribute variables.
Find candidates whose predicted probability of being hired is close to
0.5 — these are the people where the decision could easily have gone
either way. Then simulate what would happen to their prediction if their
AI bias score had been the dataset's average instead of their actual
score, holding everything else about them fixed.

## Step 16: Fit a full model predicting hired

In [25]:
# Set up the features: qualifications, AI bias score, and protected attributes
full_features = qualification_vars + ['ai_bias_score', 'is_female', 'is_non_binary', 'university_tier_num', 'is_bachelors', 'age']

X_full = hiring_df[full_features]
y_full = hiring_df['hired']

full_model = LogisticRegression(max_iter=1000, random_state=1)
full_model.fit(X_full, y_full)

LogisticRegression(max_iter=1000, random_state=1)

## Step 17: Get each candidate's predicted probability of being hired

In [26]:
# predict_proba returns two columns: probability of 0, probability of 1
# I only need the probability of being hired (column index 1)
hiring_df['predicted_prob'] = full_model.predict_proba(X_full)[:, 1]

# Review a few predictions
hiring_df[['candidate_id', 'hired', 'predicted_prob']].head()

,candidate_id,hired,predicted_prob
0,CAND_00001,0,0.015876
1,CAND_00002,1,0.988691
2,CAND_00003,1,0.781229
3,CAND_00004,1,0.890530
4,CAND_00005,0,0.094729


## Step 18: Identify the boundary candidates

I'm defining "boundary" as a predicted probability between 0.45 and 0.55
— close enough to the 0.5 cutoff that the AI score could plausibly be
the deciding factor.

In [27]:
# Filter to candidates near the decision boundary
boundary_candidates = hiring_df[
    (hiring_df['predicted_prob'] >= 0.45) & (hiring_df['predicted_prob'] <= 0.55)
].copy()

print(f"Number of boundary candidates: {len(boundary_candidates)}")

Number of boundary candidates: 249


## Step 19: Build the counterfactual scenario

For just the boundary candidates, I'm replacing their actual
`ai_bias_score` with the dataset's average score, keeping everything
else about them exactly the same, and re-predicting.

In [28]:
# Calculate the dataset's average AI bias score
average_bias_score = hiring_df['ai_bias_score'].mean()
print(f"Average AI bias score across all candidates: {round(average_bias_score, 2)}")

Average AI bias score across all candidates: 43.01


In [29]:
# Build the counterfactual feature set: same as actual, but with the average bias score
X_counterfactual = boundary_candidates[full_features].copy()
X_counterfactual['ai_bias_score'] = average_bias_score

# Get counterfactual predicted probabilities
boundary_candidates['counterfactual_prob'] = full_model.predict_proba(X_counterfactual)[:, 1]

## Step 20: Check how many candidates would flip outcome

In [30]:
# Convert both probabilities into a hire/no-hire decision using a 0.5 cutoff
boundary_candidates['actual_decision'] = (boundary_candidates['predicted_prob'] >= 0.5).astype(int)
boundary_candidates['counterfactual_decision'] = (boundary_candidates['counterfactual_prob'] >= 0.5).astype(int)

# Candidates whose decision changes between the actual and counterfactual scenario
flipped_candidates = boundary_candidates[
    boundary_candidates['actual_decision'] != boundary_candidates['counterfactual_decision']
]

print(f"Boundary candidates: {len(boundary_candidates)}")
print(f"Candidates whose outcome flips if their AI score were average: {len(flipped_candidates)}")
print(f"Percentage: {round(100 * len(flipped_candidates) / len(boundary_candidates), 1)}%")

Boundary candidates: 249
Candidates whose outcome flips if their AI score were average: 111
Percentage: 44.6%


## Step 21: Look at who those flipped candidates are

I want to see whether the flipped candidates skew toward any particular
protected attribute group, which would tie this individual-level finding
back to the aggregate patterns from Questions 1 and 2.

In [31]:
# Review the protected attributes of the candidates whose outcome flipped
flipped_candidates[['candidate_id', 'gender', 'university_tier', 'predicted_prob', 'counterfactual_prob', 'ai_bias_score']].head(10)

,candidate_id,gender,university_tier,predicted_prob,counterfactual_prob,ai_bias_score
19,CAND_00020,Male,Tier 2,0.480417,0.557238,38.6
197,CAND_00198,Female,Tier 3,0.462620,0.523869,39.5
244,CAND_00245,Male,Tier 2,0.537344,0.374067,52.5
254,CAND_00255,Male,Tier 3,0.524048,0.471708,46.0
256,CAND_00257,Female,Tier 2,0.504156,0.397262,49.2
321,CAND_00322,Female,Tier 2,0.517138,0.445724,47.1
324,CAND_00325,Non-Binary,Tier 1,0.540151,0.380008,52.3
336,CAND_00337,Female,Tier 2,0.533218,0.415218,49.8
355,CAND_00356,Female,Tier 1,0.521781,0.434786,48.0
429,CAND_00430,Male,Tier 3,0.548969,0.437577,49.4


In [32]:
# Check the gender breakdown of flipped candidates vs. all boundary candidates
print("Gender breakdown, all boundary candidates:")
print(boundary_candidates['gender'].value_counts(normalize=True).round(3))
print()
print("Gender breakdown, flipped candidates only:")
print(flipped_candidates['gender'].value_counts(normalize=True).round(3))

Gender breakdown, all boundary candidates:
gender
Male          0.522
Female        0.438
Non-Binary    0.040
Name: proportion, dtype: float64

Gender breakdown, flipped candidates only:
gender
Male          0.514
Female        0.450
Non-Binary    0.036
Name: proportion, dtype: float64


**Answer:** Out of 249 candidates sitting right at the hire/no-hire
boundary, 111 (44.6%) would have gotten a different outcome if their AI
bias score had simply been the dataset average instead of their actual
score — holding every other part of their application exactly the same.
For nearly half of the candidates in the gray zone, the AI score alone is
the deciding factor.

One nuance worth being honest about: when I checked whether the flipped
candidates skew toward any particular gender or university tier compared
to the boundary group overall, they don't — the breakdowns are nearly
identical (e.g. Female candidates are 43.8% of the boundary group and
45.0% of the flipped group). So while Questions 1 and 2 show the AI score
is systematically influenced by gender and university tier at the
aggregate level, this individual-level cut doesn't show the flips
themselves concentrated in one demographic group. The mechanism (the AI
score swinging outcomes for boundary candidates) is real and large, but
at this specific boundary-cases cut, it isn't visibly concentrated by
group — that's a limitation of this particular slice of the analysis,
not a contradiction of the earlier findings.